In [0]:
%run "../config/00_config"

In [0]:
# Testa bibliotecas necessárias para remover pasta no ADLS.

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

print("Bibliotecas carregadas com sucesso.")

In [0]:
# Cria cliente de conexão com o ADLS usando Service Principal.

from azure.core.exceptions import ResourceNotFoundError

credential = ClientSecretCredential(
    tenant_id=ADLS_TENANT_ID,
    client_id=ADLS_CLIENT_ID,
    client_secret=ADLS_CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url=f"https://{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)

file_system_client = service_client.get_file_system_client("squad3")

print("Cliente ADLS criado com sucesso.")

In [0]:
# Verifica se a pasta existe antes de excluir.

PASTA_RELATIVA = "gold/gold_ecommerce_rastreamento_entregas_volume_estado_mom"

directory_client = file_system_client.get_directory_client(PASTA_RELATIVA)

try:
    directory_client.get_directory_properties()
    print(f"Pasta encontrada: {PASTA_RELATIVA}")
except ResourceNotFoundError:
    print(f"Pasta não encontrada: {PASTA_RELATIVA}")

In [0]:
# Lista itens dentro da pasta antes da exclusão.

paths = list(file_system_client.get_paths(path=PASTA_RELATIVA, recursive=True))

print(f"Total de itens encontrados dentro da pasta: {len(paths)}")

for item in paths[:20]:
    print(item.name)

if len(paths) > 20:
    print("Exibindo apenas os 20 primeiros itens.")

In [0]:
# Exclui a pasta e todo o conteúdo interno.

for item in reversed(paths):
    if item.is_directory:
        file_system_client.get_directory_client(item.name).delete_directory()
    else:
        file_system_client.get_file_client(item.name).delete_file()

directory_client.delete_directory()

print(f"Pasta excluída: {PASTA_RELATIVA}")